# Caso 8 — Modificação de Consultas

Selecionamos 5 consultas e produzimos, manualmente, uma versão alternativa
de cada uma, cobrindo diferentes estratégias de edição: remover termos,
acrescentar termos, usar sinônimos, tornar a consulta mais específica ou
mais genérica. Cada par (original, modificada) é executado nos dois
modelos (Modelo Vetorial e BM25) usando a mesma configuração de
pré-processamento (stopwords + stemming) do restante do projeto, e
comparamos o Top-10 antes/depois.


In [ ]:
from pathlib import Path
import sys
project_root = Path.cwd()
if not (project_root / "src").is_dir() and (project_root.parent / "src").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import pandas as pd
from IPython.display import display

from src.cranfield_data import load_cranfield
from src.pre_processing import load_preprocessed, tokenize, apply_config, PREPROCESSING_CONFIGS
from src.vector_model import VectorSpaceModel
from src.bm25 import BM25

df_docs, df_queries, df_qrels = load_cranfield()
preprocessed = load_preprocessed(project_root / "data" / "processed" / "preprocessed_cranfield.pkl")

CONFIG = "stopwords_stemming"
CFG_PARAMS = PREPROCESSING_CONFIGS[CONFIG]
doc_tokens = preprocessed[CONFIG]["docs"]
query_tokens_list = preprocessed[CONFIG]["queries"]
doc_ids = df_docs["doc_id"].tolist()
query_ids = df_queries["query_id"].tolist()
doc_title = dict(zip(df_docs.doc_id, df_docs.title.str.replace("\n", " ")))
query_text = dict(zip(df_queries.query_id, df_queries.text.str.replace("\n", " ")))

vsm = VectorSpaceModel(doc_tokens)
bm25 = BM25(doc_tokens, k1=1.2, b=0.75)


def preprocess_text(raw_text):
    """Aplica a MESMA configuração de pré-processamento (stopwords + stemming)
    usada em toda a coleção a um texto de consulta arbitrário."""
    tokens = tokenize(raw_text)
    return apply_config(tokens, CFG_PARAMS["remove_stopwords"], CFG_PARAMS["apply_stemming"])

## As 5 consultas e suas versões modificadas

In [ ]:
# As 5 consultas escolhidas e suas versões modificadas manualmente,
# cobrindo as estratégias sugeridas no enunciado: remoção de termos,
# acréscimo de termos, sinônimos, e tornar mais específica/genérica.
QUERY_MODIFICATIONS = [
    {
        "query_id": "5",
        "strategy": "Remoção de termo (mais genérica)",
        "modified_text": "what chemical kinetic system is applicable to aerodynamic problems .",
    },
    {
        "query_id": "40",
        "strategy": "Substituição por sinônimos",
        "modified_text": "how can one identify transition phenomena in high-speed wakes .",
    },
    {
        "query_id": "100",
        "strategy": "Acréscimo de termo (mais específica)",
        "modified_text": (
            "what are the effects of initial imperfections on the elastic "
            "buckling of thin-walled cylindrical shells under axial compression ."
        ),
    },
    {
        "query_id": "150",
        "strategy": "Remoção de termos (mais genérica)",
        "modified_text": "what is the magnitude of second-order wing-body interference .",
    },
    {
        "query_id": "180",
        "strategy": "Acréscimo de termo (mais específica)",
        "modified_text": "how does scale height vary with altitude in an isothermal atmosphere .",
    },
]

for mod in QUERY_MODIFICATIONS:
    qid = mod["query_id"]
    print(f"Consulta {qid} [{mod['strategy']}]")
    print(f"  original : {query_text[qid]}")
    print(f"  modificada: {mod['modified_text']}")
    print()

Consulta 5 [Remoção de termo (mais genérica)]
  original : what chemical kinetic system is applicable to hypersonic aerodynamic problems .
  modificada: what chemical kinetic system is applicable to aerodynamic problems .

Consulta 40 [Substituição por sinônimos]
  original : how can one detect transition phenomena in hypersonic wakes .
  modificada: how can one identify transition phenomena in high-speed wakes .

Consulta 100 [Acréscimo de termo (mais específica)]
  original : what are the effects of initial imperfections on the elastic buckling of cylindrical shells under axial compression .
  modificada: what are the effects of initial imperfections on the elastic buckling of thin-walled cylindrical shells under axial compression .

Consulta 150 [Remoção de termos (mais genérica)]
  original : what is the magnitude of second-order wing-body interference at high supersonic mach number .
  modificada: what is the magnitude of second-order wing-body interference .

Consulta 180 [Acrésc

## Comparação de rankings: original vs. modificada

In [ ]:
def top_n_doc_ids(model, tokens, n=10):
    return [doc_ids[i] for i, _ in model.rank(tokens, top_n=n)]


def compare_query(mod, n=10):
    qid = mod["query_id"]
    original_tokens = query_tokens_list[query_ids.index(qid)]
    modified_tokens = preprocess_text(mod["modified_text"])

    print("=" * 100)
    print(f"Consulta {qid} — {mod['strategy']}")
    print(f"  Original  : {query_text[qid]!r}")
    print(f"    tokens  : {original_tokens}")
    print(f"  Modificada: {mod['modified_text']!r}")
    print(f"    tokens  : {modified_tokens}")

    grades = dict(zip(df_qrels[df_qrels.query_id == qid].doc_id,
                       df_qrels[df_qrels.query_id == qid].relevance))

    for nome, model in [("BM25", bm25), ("Modelo Vetorial", vsm)]:
        top_orig = top_n_doc_ids(model, original_tokens, n)
        top_mod = top_n_doc_ids(model, modified_tokens, n)
        overlap = len(set(top_orig) & set(top_mod))
        print(f"\n  -- {nome}: sobreposição do Top-{n} = {overlap}/{n} --")
        rows = []
        for rank in range(n):
            d_o = top_orig[rank] if rank < len(top_orig) else None
            d_m = top_mod[rank] if rank < len(top_mod) else None
            rows.append({
                "rank": rank + 1,
                "doc_id (original)": d_o,
                "grau (original)": grades.get(d_o, "-") if d_o else "-",
                "doc_id (modificada)": d_m,
                "grau (modificada)": grades.get(d_m, "-") if d_m else "-",
            })
        display(pd.DataFrame(rows).set_index("rank"))
    print()


for mod in QUERY_MODIFICATIONS:
    compare_query(mod)

Consulta 5 — Remoção de termo (mais genérica)
  Original  : 'what chemical kinetic system is applicable to hypersonic aerodynamic problems .'
    tokens  : ['chemic', 'kinet', 'system', 'applic', 'hyperson', 'aerodynam', 'problem']
  Modificada: 'what chemical kinetic system is applicable to aerodynamic problems .'
    tokens  : ['chemic', 'kinet', 'system', 'applic', 'aerodynam', 'problem']

  -- BM25: sobreposição do Top-10 = 6/10 --


,doc_id (original),grau (original),doc_id (modificada),grau (modificada)
rank,,,,
1,103,-,103,-
2,401,3,1032,-
3,1032,-,552,1
4,552,1,968,-
5,1296,1,943,-
6,968,-,401,3
7,943,-,368,-
8,625,-,746,-
9,1379,-,650,-



  -- Modelo Vetorial: sobreposição do Top-10 = 6/10 --


,doc_id (original),grau (original),doc_id (modificada),grau (modificada)
rank,,,,
1,103,-,103,-
2,552,1,1032,-
3,1032,-,552,1
4,410,-,410,-
5,367,-,367,-
6,1158,-,368,-
7,26,-,1102,-
8,1379,-,1061,-
9,401,3,968,-



Consulta 40 — Substituição por sinônimos
  Original  : 'how can one detect transition phenomena in hypersonic wakes .'
    tokens  : ['one', 'detect', 'transit', 'phenomena', 'hyperson', 'wake']
  Modificada: 'how can one identify transition phenomena in high-speed wakes .'
    tokens  : ['one', 'identifi', 'transit', 'phenomena', 'wake']

  -- BM25: sobreposição do Top-10 = 4/10 --


...

,doc_id (original),grau (original),doc_id (modificada),grau (modificada)
rank,,,,
1,1062,-1,610,-
2,1074,3,1062,-1
3,610,-,1074,3
4,1075,3,1075,3
5,672,-,672,-
6,799,-,252,-
7,687,-,799,-
8,429,-,516,-
9,252,-,520,-



Consulta 180 — Acréscimo de termo (mais específica)
  Original  : 'how does scale height vary with altitude in an atmosphere .'
    tokens  : ['scale', 'height', 'vari', 'altitud', 'atmospher']
  Modificada: 'how does scale height vary with altitude in an isothermal atmosphere .'
    tokens  : ['scale', 'height', 'vari', 'altitud', 'isotherm', 'atmospher']

  -- BM25: sobreposição do Top-10 = 10/10 --


,doc_id (original),grau (original),doc_id (modificada),grau (modificada)
rank,,,,
1,548,-1,548,-1
2,622,2,622,2
3,616,4,616,4
4,617,4,617,4
5,719,-,719,-
6,1391,-,1391,-
7,882,-,882,-
8,218,-,218,-
9,620,3,620,3



  -- Modelo Vetorial: sobreposição do Top-10 = 9/10 --


,doc_id (original),grau (original),doc_id (modificada),grau (modificada)
rank,,,,
1,548,-1,548,-1
2,622,2,622,2
3,616,4,616,4
4,314,-,314,-
5,1103,-,1103,-
6,218,-,81,-
7,613,-,218,-
8,806,-,613,-
9,617,4,806,-


## O que muda, e por quê

Resumo das sobreposições de Top 10 entre a consulta original e a
modificada:

| Consulta | Estratégia | Overlap BM25 | Overlap Vetorial |
|---|---|---|---|
| 5   | remover termo, mais genérica   | 6/10  | 6/10 |
| 40  | sinônimos                       | 4/10  | 7/10 |
| 100 | acrescentar termo, mais específica | 10/10 | 10/10 |
| 150 | remover termos, mais genérica  | 7/10  | 8/10 |
| 180 | acrescentar termo, mais específica | 10/10 | 9/10 |

Sinônimos causam a maior ruptura no BM25, na consulta 40, mas o Modelo
Vetorial é bem mais resiliente à mesma troca. Substituir `detect` por
`identify` e `hypersonic` por `high-speed` muda os tokens para
`identifi`, `high` e `speed`, totalmente diferentes dos originais
`detect` e `hyperson` do ponto de vista do modelo, já que TF-IDF e BM25
não têm nenhuma noção de sinonímia. Isso produz a menor sobreposição de
Top 10 entre todas as 10 combinações de consulta e modelo, com BM25 em
4/10. O BM25 troca 6 dos 10 primeiros colocados, e a interseção que
permanece, documentos 536, 976, 186 e 330, é sustentada pelos termos que
não mudaram, `transit`, `phenomena` e `wake`. O Modelo Vetorial, porém,
mantém 7/10, pois sua ponderação TF-IDF do documento inteiro dilui o
peso relativo dos dois termos trocados o suficiente para não desmontar o
ranking tanto quanto no BM25. Ainda assim, este continua sendo o efeito
mais visível de sinonímia dentre as 5 consultas testadas, o problema de
vocabulário dos Casos 6 e 9 provocado deliberadamente.

Acrescentar um termo coerente a uma consulta já longa e específica, na
consulta 100, não muda nada, com 10/10 em ambos os modelos. Os
documentos que já satisfaziam bem os 8 termos originais continuam no
topo, e `thin-walled` apenas reforça a ordem já estabelecida, sem
introduzir concorrentes novos fortes o bastante para desbancá-los.

Remover uma parte substancial da consulta, nas consultas 5 e 150, tem
efeito moderado, com 6/10 em ambos os modelos na consulta 5 e 7/10 no
BM25 contra 8/10 no Vetorial na consulta 150. Perder de 1 a 4 termos
elimina dimensões inteiras de casamento léxico, mas como as consultas
ainda retêm seus termos mais centrais e mais raros, boa parte do Top 10
permanece.

Acrescentar um termo a uma consulta curta quase não tem efeito na
consulta 180, mas os dois modelos reagem de forma ligeiramente
diferente. Apesar de `isotherm` ser um termo bem raro na coleção,
aparecendo em só 12 de 1400 documentos, com idf de 5,68, o Top 10 do
BM25 não muda em nada, com 10/10. Nenhum dos 12 documentos que contêm
`isotherm` também compete bem nos outros termos da consulta, `scale`,
`height`, `altitud` e `atmospher`, e por isso nenhum deles entra no Top
10 mesmo ganhando esse bônus de score. Já o Modelo Vetorial muda 1
posição, ficando em 9/10, pois sua normalização por cosseno é sensível o
suficiente a um termo de idf alto para reordenar levemente os
documentos concorrentes, mesmo sem trazer um documento novo para o Top
10. Isso mostra que o efeito de uma edição não depende só do tamanho da
consulta, mas de quão bem os documentos que contêm o termo novo também
cobrem o restante da consulta.

Padrão geral. As duas consultas com edições aditivas que preservam
todos os termos originais, 100 e 180, quase não mudam o ranking em
nenhum dos modelos. As edições subtrativas ou de substituição, 5, 40 e
150, têm efeito visível, e é justamente aí que BM25 e Modelo Vetorial
mais divergem em magnitude de reação, reforçando mais uma vez, como no
Caso 5, que a escolha do modelo importa mesmo diante da mesma edição de
consulta.
